# 01 · Exploring CUAD

Goal: understand what the supply-chain contracts in CUAD look like before we build anything.

Questions this notebook answers:
1. Which contracts are supply-chain related, and how many are there?
2. How long are they?
3. Which clause types do the lawyers' labels cover most in these contracts?
4. How are the contracts numbered? (This decides how we chunk them by clause.)

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 120)
ROOT = Path('..') / 'data' / 'raw' / 'cuad' / 'CUAD_v1'

## Load the contracts and their expert labels

`CUAD_v1.json` is in SQuAD format: each contract has its full text (`context`) and 41 questions, one per clause type. An answer is the exact span a lawyer highlighted; no answer means the clause is absent.

In [2]:
data = json.loads((ROOT / 'CUAD_v1.json').read_text(encoding='utf-8'))['data']

# The folder each PDF sits in tells us the contract type.
pdf_folder = {p.stem.lower(): p.parent.name
              for p in (ROOT / 'full_contract_pdf').rglob('*') if p.suffix.lower() == '.pdf'}

contracts, labels = [], []
for c in data:
    text = c['paragraphs'][0]['context']
    contracts.append({'title': c['title'], 'folder': pdf_folder.get(c['title'].lower(), 'Unknown'),
                      'chars': len(text), 'text': text})
    for q in c['paragraphs'][0]['qas']:
        labels.append({'title': c['title'], 'category': q['id'].split('__')[-1],
                       'answers': [a['text'] for a in q['answers']]})

contracts = pd.DataFrame(contracts)
labels = pd.DataFrame(labels)
print(f'{len(contracts)} contracts, {len(labels)} labels, {labels.category.nunique()} clause types')

510 contracts, 20910 labels, 41 clause types


## 1. Which contracts are supply-chain related?

In [3]:
SUPPLY_CHAIN = ['Supply', 'Distributor', 'Manufacturing', 'Transportation', 'Outsourcing']

def contract_type(folder):
    for t in SUPPLY_CHAIN:
        if t.lower() in folder.lower():
            return t
    return 'Other'

contracts['type'] = contracts.folder.map(contract_type)
sc = contracts[contracts.type != 'Other'].copy()
print(f'{len(sc)} supply-chain contracts')
sc.type.value_counts()

96 supply-chain contracts


type
Distributor       31
Supply            18
Outsourcing       18
Manufacturing     16
Transportation    13
Name: count, dtype: int64

## 2. How long are they?

Roughly 3,000 characters per page. Long contracts are why we can't send whole contracts to the LLM.

In [4]:
(sc.assign(pages=sc.chars / 3000)
   .groupby('type').pages.describe()[['count', 'min', '50%', 'max']].round(1))

,count,min,50%,max
type,,,,
Distributor,31.0,1.5,12.1,58.6
Manufacturing,16.0,0.6,14.7,75.0
Outsourcing,18.0,0.7,16.5,112.7
Supply,18.0,1.9,18.6,62.9
Transportation,13.0,2.0,16.4,76.6


## 3. Which clause types appear in supply-chain contracts?

`present` = share of supply-chain contracts where a lawyer found this clause. These are the questions our evaluation can test.

In [5]:
sc_labels = labels[labels.title.isin(sc.title)].copy()
sc_labels['present'] = sc_labels.answers.str.len() > 0
coverage = sc_labels.groupby('category').present.mean().sort_values(ascending=False)
coverage.round(2).to_frame('present').head(20)

,present
category,
Document Name,1.00
Parties,1.00
Agreement Date,0.93
Governing Law,0.89
Expiration Date,0.88
Effective Date,0.80
Anti-Assignment,0.75
Cap On Liability,0.66
Audit Rights,0.56


A real example: what a lawyer highlighted as the **Renewal Term** in one supply agreement.

In [6]:
example = sc_labels[(sc_labels.category == 'Renewal Term') & sc_labels.present].iloc[0]
print(example.title)
print()
print(example.answers[0])

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

If Distributor                            complies with all of the terms of this Agreement, the                            Agreement  shall be  renewable on an annual basis for                            one (1) year terms for up to  another  ten (10) years                            on the same terms and conditions as set forth herein.


## 4. How are the contracts numbered?

To chunk by clause we need to find where clauses start. Count the lines that look like a clause heading, e.g. `9.2`, `Section 4`, `ARTICLE III`.

In [7]:
HEADING = re.compile(
    r'^\s*(?:(?:section|article)\s+[\divxlc]+|\d+(?:\.\d+)*\.?\s+[A-Z])',
    re.IGNORECASE | re.MULTILINE,
)

sc['headings'] = sc.text.map(lambda t: len(HEADING.findall(t)))
print('contracts with fewer than 5 headings found:', (sc.headings < 5).sum(), 'of', len(sc))
sc.headings.describe().round(1)

contracts with fewer than 5 headings found: 17 of 96


count     96.0
mean      52.6
std       62.0
min        0.0
25%        7.8
50%       22.0
75%       83.8
max      288.0
Name: headings, dtype: float64

In [8]:
# A peek at the first headings found in one supply agreement
sample = sc[sc.type == 'Supply'].iloc[0]
print(sample.title)
for m in list(HEADING.finditer(sample.text))[:12]:
    line = sample.text[m.start():].strip().splitlines()[0]
    print('  ', line[:90])

LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-10.16_Supply Agreement
   1
   3. GOODS AND COUNTRY OF ORIGIN: 4. Specific order: The products quantity, unit price, spec
   2
   9. PORT OF DESTINATION: SHENZHEN, GUANGDONG, CHINA 10. INSURANCE: To be covered by the Sel
   3
   4
   17. FORCE MAJEURE: The Sellers shall not be held responsible for the delay in shipment or 
   5
   21. Law application It will be governed by the law of the People's Republic of China ,othe


## Takeaways

- **96 supply-chain contracts**: Distributor 31, Supply 18, Outsourcing 18, Manufacturing 16, Transportation 13.
- **Median length is 12-19 pages**, some reach 100+. Whole contracts are too big to send to the LLM, so chunking is required.
- **Clause types worth evaluating on** (present in 40%+ of these contracts): Governing Law, Expiration Date, Anti-Assignment, Cap On Liability, Audit Rights, Renewal Term, Minimum Commitment, Insurance, Warranty Duration, Termination For Convenience.
- **Heading-based chunking works for most contracts, but not all**: 17 of 96 have fewer than 5 detectable headings and need a fallback (split by paragraph / size).
- **The heading regex also catches noise**: bare page numbers ("1", "3") match. The chunker must skip lines that are only a number.
- **The text has irregular whitespace** (runs of spaces from PDF extraction). Normalize it before chunking and embedding, but keep character offsets so we can still point to the original span.